<a href="https://colab.research.google.com/github/pranathi139/GenAI/blob/main/Week2_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers torch

In [ ]:
documents = [
    "The capital of India is New Delhi.",
    "The Earth revolves around the Sun in approximately 365 days.",
    "Python is a popular programming language used for AI and data science.",
    "Google Colab is a cloud-based Jupyter notebook environment.",
    "RAG stands for Retrieval-Augmented Generation."
]


In [ ]:
def chunk_text(texts, chunk_size=50):
    return texts  # already short for demo

chunks = chunk_text(documents)


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(chunks)

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
def retrieve(query, k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [chunks[i] for i in indices[0]]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

In [ ]:
def rag_answer(question):
    context = retrieve(question)
    prompt = f"""
    Context:
    {' '.join(context)}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
questions = [
    "What is the capital of India?",
    "What does RAG stand for?",
    "How long does Earth take to orbit the Sun?",
    "What is Google Colab?",
    "What is Python?"
]

for q in questions:
    print("Q:", q)
    print("A:", rag_answer(q))
    print("-" * 50)